In [7]:
from pathlib import Path

carpeta = Path(
    "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/data/output"
)

archivos = []

for archivo in carpeta.rglob("*_clone-pass.tsv"):

    # Solo considerar archivos cuyo escenario sea A-E
    partes = archivo.stem.split("_")

    if len(partes) >= 4:

        escenario = partes[1]

        if escenario in ["A", "B", "C", "D", "E"]:
            archivos.append(archivo)

print("Archivos encontrados:", len(archivos))

Archivos encontrados: 55


In [9]:
import polars as pl

resultados = []

for archivo in archivos:

    # Leer la columna del clon
    df = pl.read_csv(
        archivo,
        separator="\t"
    ).select("clone_id")

    # Tamaño de cada clon
    clones = (
        df
        .group_by("clone_id")
        .len()
        .rename({"len": "clone_size"})
    )

    # Escenario y profundidad
    partes = archivo.stem.split("_")

    escenario = partes[1]
    profundidad = int(partes[3])

    # Métricas
    n_secuencias = df.height
    n_clones = clones.height

    singletons = (
        clones
        .filter(pl.col("clone_size") == 1)
        .height
    )

    proporcion_singletons = singletons / n_clones

    clon_mayor = clones["clone_size"].max()

    # Guardar resultados
    resultados.append({
        "escenario": escenario,
        "profundidad": profundidad,
        "n_secuencias": n_secuencias,
        "n_clones": n_clones,
        "singletons": singletons,
        "proporcion_singletons": proporcion_singletons,
        "clon_mayor": clon_mayor
    })


# Crear tabla final
resultados = pl.DataFrame(resultados)

# Ordenar
resultados = resultados.sort(
    ["escenario", "profundidad"]
)

resultados

escenario,profundidad,n_secuencias,n_clones,singletons,proporcion_singletons,clon_mayor
str,i64,i64,i64,i64,f64,i64
"""A""",100,100,100,100,1.0,1
"""A""",200,200,197,194,0.984772,2
"""A""",400,400,394,388,0.984772,2
"""A""",800,800,770,747,0.97013,5
"""A""",1600,1600,1522,1472,0.967148,6
…,…,…,…,…,…,…
"""E""",6400,6399,5233,4815,0.920122,60
"""E""",12800,12798,9140,8289,0.906893,196
"""E""",25600,25596,14805,13447,0.908274,567


In [10]:
resultados.write_csv(
    "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/clonal_table/clonal_metrics.tsv",
    separator="\t"
)